# Climate + ET time series: GridMET and PT-JPL

Assemble daily `P`, `ETo`, and ETa for each field by combining GridMET and Landsat PT-JPL ET fraction (`ETf`).

## ET/met processing chain

```
EE (PT-JPL)  →  GCS bucket  →  local sync  →  join with GridMET  →  et_join parquet
     ↑                               ↑
  Landsat                     handily sync
  collection
```

Four steps covered in this notebook:
1. **GridMET download** — daily P + ETo at field centroids via OPeNDAP
2. **PT-JPL export** — ET fraction (ETf) from Earth Engine → GCS → local
3. **Join** — interpolate ETf to daily; compute ETa = ETf × ETo
4. **Explore** — visualize seasonal patterns, ETf observations, daily AET

In [ ]:
# Shared color palette — used throughout all notebooks
STREAM_BLUE = "#2171b5"
WET_CMAP = "Blues"
TERRAIN_CMAP = "terrain"
REM_CMAP = "RdYlBu_r"
NDWI_CMAP = "RdYlGn"
STRATA_COLORS = {
    "perennial": "#2166ac",
    "intermittent": "#74add1",
    "managed": "#4dac26",
    "non_partitioned": "#d9d9d9",
}
PARTITION_COLORS = {"pe": "#a6cee3", "et_gwsm": "#1f78b4", "et_irr": "#e31a1c"}

## 1. ETo, ETf, and ETa

Handily uses:
- GridMET daily `eto` (grass reference ETo) and `prcp`
- PT-JPL ET fraction (`etf`) on Landsat overpass dates

Daily ETa is reconstructed as:

`ETa = ETf × ETo`

`ETf` is linearly interpolated between valid overpasses; inspect gaps/outliers in cloudy periods and around cuttings/harvest.

## 2. Setup

In [ ]:
import os
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from handily.config import HandilyConfig

# Load configuration
config_path = Path("beaverhead_config.toml")
with open(config_path, "rb") as f:
    cfg = tomllib.load(f)

config = HandilyConfig.from_dict(cfg)
out_dir = os.path.expanduser(config.out_dir)

print(f"GridMET parquet dir: {config.gridmet_parquet_dir}")
print(f"PT-JPL CSV dir: {config.ptjpl_csv_dir}")
print(f"Date range: {config.met_start} to {config.met_end}")

In [ ]:
# GridMET download
# download_gridmet() fetches daily P + ETo at the GridMET centroid nearest
# each field centroid. Results are cached to parquet files (one per field).
from handily.et.gridmet import download_gridmet, CLIMATE_COLS

print("CLIMATE_COLS available from GridMET:")
for col, desc in CLIMATE_COLS.items():
    print(f"  {col}: {desc}")

gridmet_dir = os.path.expanduser(config.gridmet_parquet_dir) if config.gridmet_parquet_dir else None

if config.run_met:
    fields_stratified_path = os.path.join(os.path.expanduser(config.out_dir), "fields_stratified.fgb")
    if os.path.exists(fields_stratified_path):
        import geopandas as gpd
        fields_stratified = gpd.read_file(fields_stratified_path)
        download_gridmet(
            fields=fields_stratified,
            gridmet_parquet_dir=config.gridmet_parquet_dir,
            gridmet_centroids_path=config.gridmet_centroids_path,
            start=config.met_start,
            end=config.met_end,
            feature_id=config.feature_id,
        )
        print("GridMET download complete.")
    else:
        print("[SKIP] fields_stratified.fgb not found — run stratification first")
else:
    print("[SKIP] run_met=False — loading cached GridMET parquet below")
    if gridmet_dir and os.path.exists(gridmet_dir):
        files = [f for f in os.listdir(gridmet_dir) if f.endswith('.parquet')]
        print(f"  Found {len(files)} cached parquet files in {gridmet_dir}")

In [ ]:
# Field centroids → nearest GridMET cell map (4 km grid)
import geopandas as gpd

gridmet_centroids_path = os.path.expanduser(config.gridmet_centroids_path) if config.gridmet_centroids_path else None
fields_stratified_path = os.path.join(os.path.expanduser(config.out_dir), "fields_stratified.fgb")

if (gridmet_centroids_path and os.path.exists(gridmet_centroids_path) and
        os.path.exists(fields_stratified_path)):
    centroids_gdf = gpd.read_file(gridmet_centroids_path)
    fields_strat = gpd.read_file(fields_stratified_path)
    field_centroids = fields_strat.copy()
    field_centroids["geometry"] = field_centroids.geometry.centroid

    fig, ax = plt.subplots(figsize=(10, 7))
    centroids_gdf.plot(ax=ax, color="#fdae61", markersize=30, label="GridMET cells", zorder=2)
    field_centroids.plot(ax=ax, color=STREAM_BLUE, markersize=5, label="Field centroids", zorder=3)
    fields_strat.plot(ax=ax, color="none", edgecolor="#ccc", linewidth=0.3, zorder=1)
    ax.set_title("Field Centroids → Nearest GridMET Cell (4 km grid)")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] GridMET centroids or stratified fields not available")

In [ ]:
# Seasonal patterns: ETo, precipitation, temperature
gridmet_dir = os.path.expanduser(config.gridmet_parquet_dir) if config.gridmet_parquet_dir else None

if gridmet_dir and os.path.exists(gridmet_dir):
    parquet_files = [f for f in os.listdir(gridmet_dir) if f.endswith('.parquet')]
    if parquet_files:
        import pandas as pd
        df = pd.read_parquet(os.path.join(gridmet_dir, parquet_files[0]))
        df['date'] = pd.to_datetime(df['date'])
        df['month'] = df['date'].dt.month
        n_years = df['date'].dt.year.nunique() or 1

        monthly_eto = df.groupby('month')['eto'].mean()
        monthly_prcp = df.groupby('month')['prcp'].sum() / n_years

        fig, axes = plt.subplots(2, 2, figsize=(12, 8))

        # ETo
        axes[0, 0].bar(monthly_eto.index, monthly_eto.values, color="orange", edgecolor="none", alpha=0.8)
        axes[0, 0].set_title("Mean Daily ETo (mm/day)")
        axes[0, 0].set_xlabel("Month")
        axes[0, 0].set_xticks(range(1, 13))
        axes[0, 0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])

        # Precip
        axes[0, 1].bar(monthly_prcp.index, monthly_prcp.values, color=STREAM_BLUE, edgecolor="none", alpha=0.8)
        axes[0, 1].set_title("Total Precipitation (mm/month)")
        axes[0, 1].set_xlabel("Month")
        axes[0, 1].set_xticks(range(1, 13))
        axes[0, 1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])

        # Tmax
        if 'tmax' in df.columns:
            monthly_tmax = df.groupby('month')['tmax'].mean()
            axes[1, 0].bar(monthly_tmax.index, monthly_tmax.values, color="#d73027", edgecolor="none", alpha=0.8)
            axes[1, 0].set_title("Mean Daily Tmax (°C)")
            axes[1, 0].set_xlabel("Month")
            axes[1, 0].set_xticks(range(1, 13))
            axes[1, 0].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
        else:
            axes[1, 0].text(0.5, 0.5, "tmax not in data", ha="center", va="center", transform=axes[1, 0].transAxes)

        # Tmin
        if 'tmin' in df.columns:
            monthly_tmin = df.groupby('month')['tmin'].mean()
            axes[1, 1].bar(monthly_tmin.index, monthly_tmin.values, color="#4575b4", edgecolor="none", alpha=0.8)
            axes[1, 1].set_title("Mean Daily Tmin (°C)")
            axes[1, 1].set_xlabel("Month")
            axes[1, 1].set_xticks(range(1, 13))
            axes[1, 1].set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
        else:
            axes[1, 1].text(0.5, 0.5, "tmin not in data", ha="center", va="center", transform=axes[1, 1].transAxes)

        plt.suptitle("Seasonal Climate Patterns (Beaverhead)")
        plt.tight_layout()
        plt.show()
    else:
        print("[SKIP] No GridMET parquet files found")
else:
    print("[SKIP] GridMET data not yet downloaded")

In [ ]:
# Load GridMET data if available
gridmet_dir = os.path.expanduser(config.gridmet_parquet_dir) if config.gridmet_parquet_dir else None

if gridmet_dir and os.path.exists(gridmet_dir):
    parquet_files = [f for f in os.listdir(gridmet_dir) if f.endswith('.parquet')]
    print(f"Found {len(parquet_files)} GridMET parquet files")
    
    if parquet_files:
        # Load first file as example
        example_file = os.path.join(gridmet_dir, parquet_files[0])
        df = pd.read_parquet(example_file)
        
        print(f"\nExample file: {parquet_files[0]}")
        print(f"Columns: {list(df.columns)}")
        print(f"Date range: {df['date'].min()} to {df['date'].max()}")
        print(f"Records: {len(df)}")
        
        df.head()
else:
    print(f"GridMET directory not found: {gridmet_dir}")
    print("Run the GridMET download step.")

> **Operational Callout — PT-JPL Export (Earth Engine)**
>
> Before joining, PT-JPL ETf fractions are exported from Earth Engine to GCS:
>
> ```python
> from handily.et.image_export import export_ptjpl_et_fraction
> export_ptjpl_et_fraction(
>     shapefile=config.fields_path,
>     bucket=config.et_bucket,
>     start_yr=config.ptjpl_start_yr,
>     end_yr=config.ptjpl_end_yr,
> )
> ```
>
> CLI equivalent: `handily et export --config beaverhead_config.toml`
>
> Then sync from GCS to local:
>
> ```bash
> handily sync --config beaverhead_config.toml --subdir ptjpl
> ```

## 6. Loading PT-JPL Data

In [ ]:
# Load PT-JPL data if available
ptjpl_dir = os.path.expanduser(config.ptjpl_csv_dir) if config.ptjpl_csv_dir else None

if ptjpl_dir and os.path.exists(ptjpl_dir):
    # Find CSV files recursively
    csv_files = []
    for root, dirs, files in os.walk(ptjpl_dir):
        for f in files:
            if f.endswith('.csv') and 'ptjpl' in f.lower():
                csv_files.append(os.path.join(root, f))
    
    print(f"Found {len(csv_files)} PT-JPL CSV files")
    
    if csv_files:
        # Load first file as example
        example_csv = csv_files[0]
        ptjpl_df = pd.read_csv(example_csv)
        
        print(f"\nExample file: {os.path.basename(example_csv)}")
        print(f"Columns: {list(ptjpl_df.columns)}")
        print(f"Records: {len(ptjpl_df)}")
        
        ptjpl_df.head()
else:
    print("PT-JPL CSV directory not found.")
    print("\nTo get PT-JPL data:")
    print("  1. Run EE export: python examples/beaverhead/beaverhead.py --step et")
    print("  2. Wait for EE tasks to complete (check: earthengine task list)")
    print("  3. Sync to local: handily sync --config ... --subdir ptjpl --glob '*.csv'")

In [ ]:
# PT-JPL observation calendar — which dates have valid observations?
ptjpl_dir = os.path.expanduser(config.ptjpl_csv_dir) if config.ptjpl_csv_dir else None
ptjpl_df = None
if ptjpl_dir and os.path.exists(ptjpl_dir):
    csv_files = []
    for root, dirs, files in os.walk(ptjpl_dir):
        for f in files:
            if f.endswith('.csv'):
                csv_files.append(os.path.join(root, f))
    if csv_files:
        ptjpl_df = pd.read_csv(csv_files[0])

if ptjpl_df is not None and 'date' in ptjpl_df.columns:
    ptjpl_df["date"] = pd.to_datetime(ptjpl_df["date"])
    ptjpl_df["year"] = ptjpl_df["date"].dt.year
    ptjpl_df["doy"] = ptjpl_df["date"].dt.dayofyear
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.scatter(ptjpl_df["doy"], ptjpl_df["year"], s=8, alpha=0.5, color=STREAM_BLUE)
    ax.set_xlabel("Day of Year")
    ax.set_ylabel("Year")
    ax.set_title("PT-JPL Observation Calendar (Landsat overpasses with valid ETf)")
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] PT-JPL data not available — run EE export and sync first")

### ET fraction (ETf)

ETf is unitless and can exceed 1.0 in some conditions; treat persistent spikes and long gaps as QC targets.

In [ ]:
# Visualize PT-JPL time series (if data available)
if ptjpl_dir and os.path.exists(ptjpl_dir) and csv_files:
    # Parse date if available
    if 'date' in ptjpl_df.columns:
        ptjpl_df['date'] = pd.to_datetime(ptjpl_df['date'])
        ptjpl_df = ptjpl_df.sort_values('date')
        
        # Assume etf column exists (may have different names)
        etf_cols = [c for c in ptjpl_df.columns if 'etf' in c.lower() or 'et_fraction' in c.lower()]
        if etf_cols:
            etf_col = etf_cols[0]
            
            fig, ax = plt.subplots(figsize=(14, 5))
            ax.scatter(ptjpl_df['date'], ptjpl_df[etf_col], s=20, alpha=0.7)
            ax.set_ylabel('ET Fraction')
            ax.set_xlabel('Date')
            ax.set_title('PT-JPL ET Fraction (Sparse Observations)')
            ax.axhline(1.0, color='red', linestyle='--', alpha=0.5)
            ax.set_ylim(0, 1.5)
            plt.tight_layout()
        else:
            print("ETf column not found in data")

## 7. Joining climate + ET

GridMET is daily; PT-JPL is episodic. Handily linearly interpolates ETf between valid overpasses and computes:

`ETa = ETf × ETo`

In [ ]:
# Daily AET vs ETo time series (from joined parquet, if available)
joined_dir = os.path.expanduser(config.et_join_parquet_dir) if config.et_join_parquet_dir else None
joined_df = None
if joined_dir and os.path.exists(joined_dir):
    jfiles = [f for f in os.listdir(joined_dir) if f.endswith('.parquet')]
    if jfiles:
        joined_df = pd.read_parquet(os.path.join(joined_dir, jfiles[0]))
        if 'date' in joined_df.columns:
            joined_df = joined_df.set_index(pd.to_datetime(joined_df['date']))

if joined_df is not None and "aet" in joined_df.columns:
    fig, ax = plt.subplots(figsize=(12, 4))
    ax.fill_between(joined_df.index, joined_df["aet"], alpha=0.5, color=STREAM_BLUE, label="AET")
    if "eto" in joined_df.columns:
        ax.plot(joined_df.index, joined_df["eto"], color="orange", linewidth=0.8, label="ETo")
    ax.set_ylabel("mm/day")
    ax.set_title("Daily AET vs ETo")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("[SKIP] Joined parquet not available — run the join step first")

In [ ]:
# Demonstrate interpolation concept
dates = pd.date_range('2020-04-01', '2020-09-30', freq='D')

# Simulate sparse ETf observations (every 16 days)
sparse_dates = dates[::16]
sparse_etf = np.sin(np.linspace(0, np.pi, len(sparse_dates))) * 0.7 + 0.2

# Create daily series with interpolation
etf_sparse = pd.Series(sparse_etf, index=sparse_dates)
etf_daily = etf_sparse.reindex(dates).interpolate(method='linear')

# Simulate daily ETo
eto = np.sin(np.linspace(0, np.pi, len(dates))) * 5 + 2

# Compute daily AET
aet = etf_daily * eto

# Visualize
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

# ETo
axes[0].plot(dates, eto, 'orange', linewidth=2)
axes[0].set_ylabel('ETo (mm/day)')
axes[0].set_title('Daily Reference ET (from GridMET)')

# ETf
axes[1].scatter(sparse_dates, sparse_etf, s=80, c='blue', zorder=5, label='Satellite observations')
axes[1].plot(dates, etf_daily, 'b-', alpha=0.7, label='Interpolated')
axes[1].set_ylabel('ET Fraction')
axes[1].set_title('ET Fraction (from PT-JPL)')
axes[1].legend()

# AET
axes[2].fill_between(dates, 0, aet, alpha=0.5, color='green')
axes[2].plot(dates, aet, 'g-', linewidth=1)
axes[2].set_ylabel('AET (mm/day)')
axes[2].set_xlabel('Date')
axes[2].set_title('Actual ET = ETf × ETo')

plt.tight_layout()

In [ ]:
from handily.et.join import join_gridmet_ptjpl

if config.run_join:
    join_gridmet_ptjpl(
        gridmet_parquet_dir=config.gridmet_parquet_dir,
        ptjpl_csv_dir=config.ptjpl_csv_dir,
        out_parquet_dir=config.et_join_parquet_dir,
        ptjpl_csv_template=config.ptjpl_csv_template,
        feature_id=config.feature_id,
    )
    print("ET join complete.")
else:
    print("[SKIP] run_join=False — loading cached joined parquet below")
    joined_dir = os.path.expanduser(config.et_join_parquet_dir) if config.et_join_parquet_dir else None
    if joined_dir and os.path.exists(joined_dir):
        files = [f for f in os.listdir(joined_dir) if f.endswith('.parquet')]
        print(f"  Found {len(files)} cached joined parquet files")

In [ ]:
# Check if joined data exists
joined_dir = os.path.expanduser(config.et_join_parquet_dir) if config.et_join_parquet_dir else None

if joined_dir and os.path.exists(joined_dir):
    parquet_files = [f for f in os.listdir(joined_dir) if f.endswith('.parquet')]
    print(f"Found {len(parquet_files)} joined parquet files")
    
    if parquet_files:
        example_file = os.path.join(joined_dir, parquet_files[0])
        joined_df = pd.read_parquet(example_file)
        
        print(f"\nExample file: {parquet_files[0]}")
        print(f"Columns: {list(joined_df.columns)}")
        print(f"Date range: {joined_df['date'].min()} to {joined_df['date'].max()}")
        
        joined_df.head()
else:
    print("Joined parquet directory not found.")
    print("Run the ET join step.")

> **Repo Capability — Notebook 4 (Climate and ET)**
> Modules exercised: `handily.et.gridmet`, `handily.et.image_export`, `handily.et.join`, `handily.bucket`
> Key functions: `download_gridmet()`, `export_ptjpl_et_fraction()`, `join_gridmet_ptjpl()`
> CLI equivalents: `handily met download`, `handily et export`, `handily et join`, `handily sync --subdir ptjpl`

> **Artifacts Produced**
> - `{gridmet_parquet_dir}/{FID}.parquet` — daily GridMET (eto, prcp, tmax, tmin) per field
> - `{ptjpl_csv_dir}/{FID}/ptjpl_etf_zonal_{FID}_*.csv` — PT-JPL ETf per overpass date
> - `{et_join_parquet_dir}/{FID}.parquet` — joined daily time series (eto, prcp, etf, aet)

### Output Files

The join step produces parquet files with combined time series:

| Column | Description |
|--------|-------------|
| `date` | Date |
| `eto` | Reference ET (mm/day) |
| `prcp` | Precipitation (mm/day) |
| `etf` | ET fraction (interpolated) |
| `aet` | Actual ET = etf × eto (mm/day) |

## 9. Exploring Joined Data

In [ ]:
# Compare ETo, ETf, and AET
if joined_dir and os.path.exists(joined_dir) and parquet_files:
    joined_df['date'] = pd.to_datetime(joined_df['date'])
    
    fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
    
    # ETo
    if 'eto' in joined_df.columns:
        axes[0].plot(joined_df['date'], joined_df['eto'], 'orange', alpha=0.7)
        axes[0].set_ylabel('ETo (mm/day)')
        axes[0].set_title('Reference ET')
    
    # ETf
    if 'etf' in joined_df.columns:
        axes[1].plot(joined_df['date'], joined_df['etf'], 'blue', alpha=0.7)
        axes[1].set_ylabel('ETf')
        axes[1].set_title('ET Fraction (Interpolated)')
        axes[1].set_ylim(0, 1.5)
    
    # AET
    if 'aet' in joined_df.columns:
        axes[2].fill_between(joined_df['date'], 0, joined_df['aet'], alpha=0.5, color='green')
        axes[2].set_ylabel('AET (mm/day)')
        axes[2].set_title('Actual ET')
    
    axes[2].set_xlabel('Date')
    plt.tight_layout()

## Key takeaways

- Daily ETa is reconstructed from PT-JPL ETf and GridMET ETo.
- Linear interpolation is a pragmatic gap-fill; review ETf behavior during cloudy periods.
- PT-JPL exports are Earth Engine → GCS → local mirror via `handily sync`.

**Next**: [Notebook 05 - ET Partitioning](05_et_partitioning.ipynb)